# Data Mining Assignment – Spark Data Lake Preprocessing

**Student:** Lavesh Jadon (MSD25023)
**Dataset:** `sales.csv` | **Environment:** Google Colab + PySpark

Bronze layer = raw `sales.csv`, kept byte-for-byte unchanged.
Silver layer = basic preprocessing with Apache Spark, saved as CSV.

## 0. Install PySpark

In [ ]:
%pip install -q pyspark

## 1. Data Lake structure
```
data_lake/
|-- bronze/
|   `-- sales.csv
`-- silver/
    `-- preprocessed_sales/
```
`sales.csv` is taken from the working directory (upload it in Colab), from the repository's `dataset/` folder, or downloaded from the class repository. It is copied as raw bytes, so the Bronze file is identical to the source.

In [ ]:
import os, shutil, hashlib, urllib.request

BRONZE_DIR = "data_lake/bronze"
SILVER_DIR = "data_lake/silver/preprocessed_sales"
BRONZE_FILE = f"{BRONZE_DIR}/sales.csv"
RAW_URL = "https://raw.githubusercontent.com/necromancersanju-netizen/data-mining-data-lake-assignment/main/dataset/sales.csv"

os.makedirs(BRONZE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(SILVER_DIR), exist_ok=True)

if not os.path.exists(BRONZE_FILE):
    local = next((p for p in ["sales.csv", "../../dataset/sales.csv", "dataset/sales.csv"] if os.path.exists(p)), None)
    if local:
        shutil.copyfile(local, BRONZE_FILE)          # byte-exact copy, no modification
    else:
        urllib.request.urlretrieve(RAW_URL, BRONZE_FILE)

md5 = lambda p: hashlib.md5(open(p, "rb").read()).hexdigest()
print("Bronze file:", BRONZE_FILE, "| size:", os.path.getsize(BRONZE_FILE), "bytes | md5:", md5(BRONZE_FILE))
for root, dirs, files in os.walk("data_lake"):
    print(root, files)

## 2. Load and inspect with Spark

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .appName("SalesDataLake")
         .config("spark.sql.ansi.enabled", "false")                 # invalid casts/dates -> null instead of error
         .config("spark.sql.legacy.timeParserPolicy", "CORRECTED")  # strict date parsing (31/02 is invalid)
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
spark.version

In [ ]:
# Read every column as string so raw, malformed values can be inspected
raw_df = spark.read.csv(BRONZE_FILE, header=True, inferSchema=False)
raw_df.printSchema()
raw_df.show(5, truncate=False)
raw_count = raw_df.count()
print("Total records in Bronze:", raw_count)

### 2.1 Duplicates

In [ ]:
exact_dups = raw_count - raw_df.dropDuplicates().count()
print("Exact duplicate rows:", exact_dups)

dup_ids = raw_df.dropDuplicates().groupBy("order_id").count().filter("count > 1")
print("order_id values shared by different (non-identical) rows:", dup_ids.count())
raw_df.dropDuplicates().join(dup_ids.select("order_id"), "order_id").orderBy("order_id").show(6, truncate=False)

### 2.2 Null / blank values

In [ ]:
raw_df.select([
    F.sum((F.col(c).isNull() | (F.trim(F.col(c)) == "")).cast("int")).alias(c)
    for c in raw_df.columns
]).show(truncate=False)

### 2.3 Text-format inconsistencies (extra spaces, capitalization)

In [ ]:
TEXT_COLS = ["customer_name", "product", "category", "payment_method", "city", "state", "order_status"]
for c in TEXT_COLS:
    v = F.col(c)
    n_space = raw_df.filter(v != F.trim(v)).count()
    # values that are equal after trim + lower-casing but are written differently
    variants = (raw_df.filter(v.isNotNull())
                .groupBy(F.lower(F.trim(v)).alias("key"))
                .agg(F.collect_set(c).alias("variants"))
                .filter(F.size("variants") > 1))
    n_var = variants.count()
    print(f"{c}: values with extra spaces = {n_space}, values with case/space variants = {n_var}")
    if n_var:
        variants.show(truncate=False)

### 2.4 Invalid numerical values

In [ ]:
q = raw_df.select(
    F.col("quantity").cast("int").alias("quantity"),
    F.col("unit_price").cast("double").alias("unit_price"),
    F.col("discount_percent").cast("double").alias("discount_percent"))
checks = {
    "quantity not numeric":           F.col("quantity").isNull(),
    "quantity <= 0":                  F.col("quantity") <= 0,
    "unit_price not numeric":         F.col("unit_price").isNull(),
    "unit_price <= 0":                F.col("unit_price") <= 0,
    "discount not numeric":           F.col("discount_percent").isNull(),
    "discount outside 0-100":         (F.col("discount_percent") < 0) | (F.col("discount_percent") > 100),
}
q.select([F.sum(cond.cast("int")).alias(name) for name, cond in checks.items()]).show(truncate=False)
q.describe().show()

### 2.5 Inconsistent / invalid dates

In [ ]:
DATE_FORMATS = ["yyyy-MM-dd", "dd/MM/yyyy", "yyyy/MM/dd", "MM-dd-yyyy"]
d = F.trim(F.col("order_date"))
raw_df.groupBy(F.regexp_replace(d, "[0-9]", "9").alias("date_pattern")).count().orderBy(F.desc("count")).show(truncate=False)

parsed = F.coalesce(*[F.to_date(d, f) for f in DATE_FORMATS])
non_iso = ~d.rlike(r"^\d{4}-\d{2}-\d{2}$")
print("Dates not in yyyy-MM-dd format:", raw_df.filter(non_iso).count())
print("Dates that are not a real calendar date in any known format:", raw_df.filter(parsed.isNull()).count())
raw_df.filter(non_iso | parsed.isNull()).select("order_id", "order_date").show(truncate=False)

## 3. Basic preprocessing – Silver layer

In [ ]:
# 1) Remove exact duplicate rows
df = raw_df.dropDuplicates()
print("After removing exact duplicates:", df.count())

# 3) Trim spaces in all fields; empty strings become null
for c in df.columns:
    df = df.withColumn(c, F.when(F.trim(F.col(c)) == "", None).otherwise(F.trim(F.col(c))))

# Standardise obvious capitalization issues (Title Case) where that is safe
for c in ["customer_name", "category", "city", "state"]:
    df = df.withColumn(c, F.initcap(F.col(c)))

In [ ]:
# 4) Numeric validation: cast types, mark clearly invalid values as null
df = (df.withColumn("quantity", F.col("quantity").cast("int"))
        .withColumn("unit_price", F.col("unit_price").cast("double"))
        .withColumn("discount_percent", F.col("discount_percent").cast("double"))
        .withColumn("quantity", F.when(F.col("quantity") > 0, F.col("quantity")))
        .withColumn("unit_price", F.when(F.col("unit_price") > 0, F.col("unit_price")))
        .withColumn("discount_percent", F.when(F.col("discount_percent").between(0, 100), F.col("discount_percent"))))

# 5) Dates: parse the known formats into one DateType (yyyy-MM-dd); impossible dates become null
df = df.withColumn("order_date", F.coalesce(*[F.to_date(F.col("order_date"), f) for f in DATE_FORMATS]))

print("Nulls after validation:")
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show(truncate=False)

In [ ]:
# 2) Handle missing values, keeping the records instead of dropping them:
#    unit_price       -> median price of the same product
#    quantity         -> overall median quantity
#    discount_percent -> overall median discount
#    text fields      -> "Unknown"
#    order_date       -> left null (an invented date would be misleading)
product_median = df.groupBy("product").agg(F.percentile_approx("unit_price", 0.5).alias("product_median_price"))
qty_median, disc_median = df.select(F.percentile_approx("quantity", 0.5),
                                    F.percentile_approx("discount_percent", 0.5)).first()
print("Median quantity:", qty_median, "| median discount:", disc_median)

silver_df = (df.join(product_median, "product", "left")
               .withColumn("unit_price", F.coalesce("unit_price", "product_median_price"))
               .fillna({"quantity": qty_median, "discount_percent": disc_median,
                        "customer_name": "Unknown", "payment_method": "Unknown", "city": "Unknown"})
               .select(raw_df.columns)
               .orderBy("order_id"))

silver_df.printSchema()
print("Nulls in Silver:")
silver_df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in silver_df.columns]).show(truncate=False)

In [ ]:
# Save the Silver layer as a single CSV part file
silver_df.coalesce(1).write.mode("overwrite").option("header", True).csv(SILVER_DIR)

# Copy the part file to a readable name for submission: silver/sales_silver.csv
part = next(f for f in os.listdir(SILVER_DIR) if f.startswith("part-") and f.endswith(".csv"))
os.makedirs("silver", exist_ok=True)
shutil.copyfile(os.path.join(SILVER_DIR, part), "silver/sales_silver.csv")
print(sorted(os.listdir(SILVER_DIR)))

## 4. Verification

In [ ]:
silver_read = spark.read.csv(SILVER_DIR, header=True, inferSchema=True)
silver_count = silver_read.count()
print("Rows before preprocessing (Bronze):", raw_count)
print("Rows after preprocessing  (Silver):", silver_count)
print("Rows removed (exact duplicates):", raw_count - silver_count)
silver_read.printSchema()
silver_read.show(10, truncate=False)

In [ ]:
# Re-run the discrepancy checks on the Silver data
print("Exact duplicates:", silver_count - silver_read.dropDuplicates().count())
print("quantity <= 0:", silver_read.filter("quantity <= 0").count())
print("unit_price <= 0:", silver_read.filter("unit_price <= 0").count())
print("discount outside 0-100:", silver_read.filter("discount_percent < 0 OR discount_percent > 100").count())
print("Invalid dates kept as null:", silver_read.filter(F.col("order_date").isNull()).count())
silver_read.groupBy("category").count().orderBy("category").show()
silver_read.groupBy("city").count().orderBy("city").show()
print("Bronze md5 (unchanged):", md5(BRONZE_FILE))

### Preprocessing performed
1. **Duplicates** – exact duplicate rows removed with `dropDuplicates()`. Rows that share an `order_id` but differ in every other field are separate orders and were kept.
2. **Text cleanup** – leading/trailing spaces trimmed in all columns, empty strings turned into null, Title Case applied to `customer_name`, `category`, `city`, `state` (e.g. `ELECTRONICS`, `delhi`, ` Mumbai `). `product` and `payment_method` were already consistent and left as is, so values like `USB-C Hub` stay correct.
3. **Numeric validation** – `quantity` ≤ 0, `unit_price` ≤ 0 and `discount_percent` outside 0–100 treated as invalid (set to null).
4. **Missing / invalid values** – `unit_price` filled with the median price of the same product; `quantity` and `discount_percent` with the overall median; missing `customer_name`, `payment_method`, `city` set to `Unknown`.
5. **Dates** – `yyyy-MM-dd`, `dd/MM/yyyy`, `yyyy/MM/dd`, `MM-dd-yyyy` converted to one format (`yyyy-MM-dd`); impossible values (`31/02/2026`, `not-a-date`) set to null while the order record is kept.
6. **Output** – Silver data saved as CSV in `data_lake/silver/preprocessed_sales/` and copied to `silver/sales_silver.csv`. The Bronze file is unchanged.

In [ ]:
spark.stop()